In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes
!pip install -q pillow numpy opencv-python matplotlib scikit-image peft

import torch
print(f" PyTorch version: {torch.__version__}")
print(f" GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
 PyTorch version: 2.10.0+cu128
 GPU Available: True
 GPU: Tesla T4


In [5]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image
import numpy as np
import json
from typing import Tuple
import torch

class LMMModelLoader:
    def __init__(self, model_name: str = "Qwen/Qwen2-VL-7B-Instruct"):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self.processor = None
        print(f" Using device: {self.device}")

    def load_model(self):
        from transformers import BitsAndBytesConfig
        print(f" Loading {self.model_name}...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        self.processor = AutoProcessor.from_pretrained(
            self.model_name,
            trust_remote_code=True
        )

        print(f" Model loaded successfully!")
        return self.model, self.processor

    def inference(self, image: Image.Image, question: str, max_tokens: int = 100) -> str:
        if self.model is None:
            self.load_model()

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": question},
                ],
            }
        ]

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens
            )

        generated_ids = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, output_ids)
        ]
        response = self.processor.batch_decode(
            generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )[0]

        return response.strip()

print("\n Initializing model loader...")
loader = LMMModelLoader(model_name="Qwen/Qwen2-VL-7B-Instruct")
print(" Ready to load model on first use")


 Initializing model loader...
 Using device: cuda
 Ready to load model on first use


In [6]:
import cv2
from skimage import exposure

class PerturbationAttack:
    """Pixel-level perturbation attacks"""

    @staticmethod
    def gaussian_noise(image: Image.Image, epsilon: float = 25) -> Image.Image:
        """Add Gaussian noise"""
        img_array = np.array(image, dtype=np.float32)
        noise = np.random.normal(0, epsilon, img_array.shape)
        perturbed = np.clip(img_array + noise, 0, 255)
        return Image.fromarray(perturbed.astype(np.uint8))

    @staticmethod
    def brightness_contrast(image: Image.Image, factor: float = 1.5) -> Image.Image:
        """Modify brightness and contrast"""
        img_array = np.array(image, dtype=np.float32) / 255.0
        stretched = exposure.equalize_adapthist(img_array)
        adjusted = np.clip(stretched * factor, 0, 1)
        return Image.fromarray((adjusted * 255).astype(np.uint8))

    @staticmethod
    def color_shift(image: Image.Image) -> Image.Image:
        """Shift color channels"""
        img_array = np.array(image)
        if len(img_array.shape) == 3:
            shifted = np.roll(img_array, 50, axis=2)
            return Image.fromarray(np.clip(shifted, 0, 255).astype(np.uint8))
        return image

    @staticmethod
    def blur_attack(image: Image.Image, kernel_size: int = 21) -> Image.Image:
        """Apply blur to confuse model"""
        img_array = np.array(image)
        blurred = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0)
        return Image.fromarray(blurred.astype(np.uint8))

class PatchAttack:
    """Adversarial patch attacks"""

    @staticmethod
    def random_patch(image: Image.Image, patch_size: int = 64) -> Tuple[Image.Image, dict]:
        """Add random colored patch"""
        img_array = np.array(image, dtype=np.float32)
        height, width = img_array.shape[:2]

        top = np.random.randint(0, max(1, height - patch_size))
        left = np.random.randint(0, max(1, width - patch_size))

        patch = np.random.randint(0, 256, (patch_size, patch_size, 3), dtype=np.uint8)

        img_copy = img_array.copy()
        img_copy[top:top+patch_size, left:left+patch_size] = patch

        metadata = {"top": int(top), "left": int(left), "size": patch_size}
        return Image.fromarray(np.clip(img_copy, 0, 255).astype(np.uint8)), metadata

    @staticmethod
    def white_patch(image: Image.Image, patch_size: int = 64) -> Tuple[Image.Image, dict]:
        """Add white square patch"""
        img_array = np.array(image, dtype=np.float32)
        height, width = img_array.shape[:2]

        top = np.random.randint(0, max(1, height - patch_size))
        left = np.random.randint(0, max(1, width - patch_size))

        img_copy = img_array.copy()
        img_copy[top:top+patch_size, left:left+patch_size] = 255

        metadata = {"top": int(top), "left": int(left), "size": patch_size}
        return Image.fromarray(np.clip(img_copy, 0, 255).astype(np.uint8)), metadata

print("Attack classes ready")

Attack classes ready


In [7]:
from difflib import SequenceMatcher

class AttackEvaluator:
    """Evaluate attack success"""

    def __init__(self):
        self.results = []
        self.success_count = 0
        self.total_attacks = 0

    @staticmethod
    def text_similarity(text1: str, text2: str) -> float:
        """Calculate text similarity (0-1)"""
        return SequenceMatcher(None, text1.lower(), text2.lower()).ratio()

    @staticmethod
    def is_successful(original: str, attacked: str, threshold: float = 0.6) -> bool:
        """Check if attack was successful"""
        similarity = AttackEvaluator.text_similarity(original, attacked)
        return similarity < threshold  # Less similar = successful attack

    def evaluate(self, image_name: str, attack_type: str,
                 original_output: str, attacked_output: str, question: str) -> dict:
        """Evaluate single attack"""

        similarity = self.text_similarity(original_output, attacked_output)
        success = self.is_successful(original_output, attacked_output)

        result = {
            "image_name": image_name,
            "attack_type": attack_type,
            "question": question[:100],  # Truncate
            "original_output": original_output[:150],
            "attacked_output": attacked_output[:150],
            "similarity": round(similarity, 3),
            "success": success
        }

        self.results.append(result)
        self.total_attacks += 1
        if success:
            self.success_count += 1

        return result

    def print_summary(self):
        """Print statistics"""
        if self.total_attacks == 0:
            return

        success_rate = (self.success_count / self.total_attacks) * 100

        print("\n" + "="*60)
        print(" ATTACK RESULTS SUMMARY")
        print("="*60)
        print(f"Total Attacks: {self.total_attacks}")
        print(f"Successful Attacks: {self.success_count}")
        print(f"Success Rate: {success_rate:.1f}%")
        print("="*60)

    def save(self, filepath: str = "/content/attack_results.json"):
        """Save results"""
        with open(filepath, 'w') as f:
            json.dump({
                "total": self.total_attacks,
                "successful": self.success_count,
                "success_rate": round(self.success_count/self.total_attacks*100, 1) if self.total_attacks > 0 else 0,
                "results": self.results
            }, f, indent=2)
        print(f"Results saved to {filepath}")

evaluator = AttackEvaluator()
print(" Evaluator ready")

 Evaluator ready


In [8]:
import time

class AttackPipeline:
    """Execute attacks end-to-end"""

    def __init__(self, loader: LMMModelLoader, evaluator: AttackEvaluator):
        self.loader = loader
        self.evaluator = evaluator
        self.perturbation = PerturbationAttack()
        self.patch = PatchAttack()

    def run_attack(self, image: Image.Image, question: str,
                   attack_type: str, image_name: str = "test") -> Tuple[dict, Image.Image]:
        """Execute single attack"""

        print(f"\n Running {attack_type} attack...")

        # Get original output
        original_output = self.loader.inference(image, question)
        print(f"Original: {original_output[:80]}...")

        # Generate attacked image
        attacked_image = None

        if attack_type == "gaussian_noise":
            attacked_image = self.perturbation.gaussian_noise(image, epsilon=30)
        elif attack_type == "brightness_contrast":
            attacked_image = self.perturbation.brightness_contrast(image, factor=2.0)
        elif attack_type == "color_shift":
            attacked_image = self.perturbation.color_shift(image)
        elif attack_type == "blur":
            attacked_image = self.perturbation.blur_attack(image, kernel_size=15)
        elif attack_type == "random_patch":
            attacked_image, _ = self.patch.random_patch(image, patch_size=80)
        elif attack_type == "white_patch":
            attacked_image, _ = self.patch.white_patch(image, patch_size=100)

        if attacked_image is None:
            print(f" Unknown attack type: {attack_type}")
            return None, None

        # Get attacked output
        attacked_output = self.loader.inference(attacked_image, question)
        print(f"Attacked: {attacked_output[:80]}...")

        # Evaluate
        result = self.evaluator.evaluate(
            image_name, attack_type, original_output, attacked_output, question
        )

        status = " SUCCESS" if result["success"] else "❌ FAILED"
        print(f"{status} | Similarity: {result['similarity']}")

        return result, attacked_image

    def batch_attack(self, image: Image.Image, question: str,
                    attack_types: list = None, image_name: str = "test") -> dict:
        """Run multiple attacks"""

        if attack_types is None:
            attack_types = ["gaussian_noise", "brightness_contrast", "color_shift",
                          "blur", "random_patch", "white_patch"]

        results = {}

        print(f"\n{'='*60}")
        print(f" BATCH ATTACK: {image_name}")
        print(f"Attacks: {len(attack_types)} | Question: {question[:50]}...")
        print(f"{'='*60}")

        for i, attack in enumerate(attack_types, 1):
            print(f"\n[{i}/{len(attack_types)}] {attack}")
            try:
                result, img = self.run_attack(image, question, attack, image_name)
                if result:
                    results[attack] = result
                time.sleep(1)  # Rate limit
            except Exception as e:
                print(f" Error: {e}")

        return results

pipeline = AttackPipeline(loader, evaluator)
print("Pipeline ready")

Pipeline ready


In [9]:
import os
import requests
from PIL import Image
from io import BytesIO

os.makedirs('/content/test_images', exist_ok=True)

test_images = {
    'cat': 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=512',
    'dog': 'https://images.unsplash.com/photo-1552053831-71594a27632d?w=512',
    'bird': 'https://images.unsplash.com/photo-1444464666168-49d633b86797?w=512'
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print(" Downloading test images...")
for name, url in test_images.items():
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img.save(f'/content/test_images/{name}.jpg')
        print(f" {name} — {img.size}")
    except Exception as e:
        print(f" {name}: {e}")

print("\n Images ready")

 cat — (512, 352)
 dog — (512, 821)
 bird — (512, 342)

 Images ready


In [10]:
print(" Loading model (this may take 2-3 mins)...")
model, processor = loader.load_model()
print(" Model loaded!\n")

# Run attack on one image
image_path = '/content/test_images/cat.jpg'

if os.path.exists(image_path):
    test_image = Image.open(image_path).resize((512, 512))
    question = "What animal is in this image? Describe it briefly."

    results = pipeline.batch_attack(
        test_image,
        question,
        image_name="cat_demo"
    )

    # Show results
    evaluator.print_summary()
else:
    print("Test image not found")

 Loading model (this may take 2-3 mins)...
 Loading Qwen/Qwen2-VL-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 Model loaded successfully!
 Model loaded!


 BATCH ATTACK: cat_demo
Attacks: 6 | Question: What animal is in this image? Describe it briefly....

[1/6] gaussian_noise

 Running gaussian_noise attack...
Original: The image shows a cat. The cat has a black and white coat with green eyes. It ap...
Attacked: The image shows a cat. The cat has a black and white coat with green eyes. It ap...
❌ FAILED | Similarity: 0.809

[2/6] brightness_contrast

 Running brightness_contrast attack...
Original: The image shows a cat. The cat has a black and white coat with green eyes. It ap...
Attacked: The image shows a cat. The cat has a white and black coat with a blue patch on i...
❌ FAILED | Similarity: 0.712

[3/6] color_shift

 Running color_shift attack...
Original: The image shows a cat. The cat has a black and white coat with green eyes. It ap...
Attacked: The image shows a cat. The cat has a white body with black markings on its face,...
❌ FAILED | Similarity: 0.652

[4/6] blur

 Running blur a

In [11]:
evaluator.save('/content/attack_results.json')

# Also save as CSV
import pandas as pd
df = pd.DataFrame(evaluator.results)
df.to_csv('/content/attack_results.csv', index=False)
print(" CSV saved: /content/attack_results.csv")

print("\n Results Summary:")
print(df[['image_name', 'attack_type', 'success']].to_string())

Results saved to /content/attack_results.json
 CSV saved: /content/attack_results.csv

 Results Summary:
  image_name          attack_type  success
0   cat_demo       gaussian_noise    False
1   cat_demo  brightness_contrast    False
2   cat_demo          color_shift    False
3   cat_demo                 blur    False
4   cat_demo         random_patch    False
5   cat_demo          white_patch    False


In [12]:
import matplotlib.pyplot as plt

# Get first successful attack for visualization
successful_attacks = [r for r in evaluator.results if r['success']]

if successful_attacks:
    # Re-run to get images
    test_image = Image.open('/content/test_images/cat.jpg').resize((512, 512))

    _, attacked_img = pipeline.run_attack(
        test_image,
        question,
        successful_attacks[0]['attack_type'],
        "visualization"
    )

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(test_image)
    axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(attacked_img)
    axes[1].set_title(f"After {successful_attacks[0]['attack_type']}", fontsize=14, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig('/content/attack_visualization.png', dpi=150, bbox_inches='tight')
    print(" Visualization saved!")
    plt.show()

In [14]:
from google.colab import files

print(" Preparing download...")
files.download('/content/attack_results.json')
files.download('/content/attack_results.csv')
if os.path.exists('/content/attack_visualization.png'):
    files.download('/content/attack_visualization.png')

print(" Files ready for download!")

 Preparing download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Files ready for download!
